# Tutorial: Cross-Event Generalization and Domain Shift

**Audience**
- Readers who already understand the explanatory mainline and now want to know why transport is harder.

**Prerequisites**
- The first notebook.
- Basic familiarity with holdout validation.

**Learning goals**
- Understand how the repo builds event-level context.
- Read the leave-one-event-out transport loop.
- See why stabilization still does not fully solve domain shift.


## Outline

1. Why this notebook exists.
2. Event profiles are explicit code, not just metadata.
3. V3 build and stabilization.
4. The figures that reveal domain shift.
5. What this notebook should leave you with.


In [ ]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
MODELING_DIR = REPO_ROOT / "project" / "modeling"
OUTPUT_DIR = MODELING_DIR / "output"


def load_json(rel_path: str):
    return json.loads((REPO_ROOT / rel_path).read_text(encoding="utf-8"))


def load_csv(rel_path: str) -> pd.DataFrame:
    return pd.read_csv(REPO_ROOT / rel_path)


def get_def_source(rel_path: str, name: str, max_lines: int = 80) -> str:
    source = (REPO_ROOT / rel_path).read_text(encoding="utf-8")
    tree = ast.parse(source)
    lines = source.splitlines()
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
            block = lines[node.lineno - 1 : node.end_lineno]
            if len(block) > max_lines:
                block = block[:max_lines] + ["# ... truncated for notebook readability ..."]
            return "\n".join(block)
    raise KeyError(f"{name} was not found in {rel_path}")


def print_defs(rel_path: str, *names: str, max_lines: int = 80) -> None:
    for name in names:
        print(f"\n===== {name} ({rel_path}) =====\n")
        print(get_def_source(rel_path, name, max_lines=max_lines))


print("Repository root:", REPO_ROOT)
print("Modeling directory:", MODELING_DIR)


## 1. Why this notebook exists

The explanatory line can look solid while cross-event transport still fails.

This notebook asks:

> If the model has never seen one event before, can it still rank damage or recovery on that held-out event?

That is why event profiles, fold-safe preprocessing, and event-level shift diagnostics are central here.


In [ ]:
event_profile = load_csv('project/modeling/pixel_data/event_profile_v1.csv')
cross_event_summary = load_csv('project/modeling/output/model_summary_cross_event_v3.csv')

display(event_profile[['event_id', 'disaster_type', 'event_duration_days', 'storm_precip_7d', 'quality_flag']])
display(cross_event_summary[['model', 'scope', 'metric_name', 'value', 'baseline_value', 'delta_vs_baseline', 'notes']])


## 2. Event profiles are explicit code, not just metadata

In [ ]:
print_defs(
    'project/modeling/pipelines/02_cross_event_pipeline.py',
    '_build_event_profile',
    '_attach_event_features',
    '_run_loeo',
    max_lines=90,
)


The important idea is that the transport line tries to stay interpretable while acknowledging event heterogeneity.

Instead of hiding event differences with a random split, it makes those differences visible and then tests whether the model survives them.


## 3. V3 build and stabilization

In [ ]:
print_defs(
    'project/modeling/pipelines/02_cross_event_pipeline.py',
    '_build_v3_impl',
    '_stabilize_run_loeo_round',
    'stabilize_main',
    max_lines=90,
)


## Current cross-event reading

- V3 Logit `AUC = 0.4897`.
- V3 Logit `Brier = 0.3110`.
- V3 AFT `c_index = 0.5354`.
- V3 Cox `c_index = 0.4641`.
- Stabilization stopped at `r1` with reason `marginal_improvement`.

This file is about a real domain-shift problem, not a cosmetic pipeline mismatch.


## 4. The figures that reveal domain shift

In [ ]:
shift = load_csv('project/modeling/output/cross_event_shift_diagnostics_v3.csv')
rounds = load_csv('project/modeling/output/cross_event_round_comparison_v3x.csv')
stop = load_json('project/modeling/output/cross_event_stop_decision_v3x.json')

display(rounds[['round_id', 'metric_name', 'value', 'delta', 'stop_flag']])
stop


## Figures to read

### Pairwise shift heatmap
![](../figures/cross_event/shift_pairwise_smd_heatmap_v3.png)

### Transport metrics vs anchor
![](../figures/cross_event/transport_metrics_compare_v3.png)

### Benchmark importance view
![](../figures/cross_event/hgb_permutation_importance_v3.png)


## 5. What this notebook should leave you with

- The transport problem is harder because event distributions really differ.
- Leave-one-event-out validation exposes that difference directly.
- Stabilization improves some metrics, but the stop rule still says the gain is marginal.


## Optional reader exercise

Pick one event from the shift heatmap and explain why its profile might be hard to transfer.

In [ ]:
# Exercise scaffold: inspect one metric family.
rounds[rounds['metric_group'] == 'classification']
